<a href="https://colab.research.google.com/github/mejia080902-bit/simulacion_II/blob/main/simulacion_II_tarea_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
Simulación II - Variables antitéticas vs. métodos base
Integral objetivo: ∫_0^1 sqrt(arctan(x)) dx
"""

import numpy as np
import random as rd
import time

# -------------------------
# Función de interés
# -------------------------
def g(x):
    """g(x) = sqrt(arctan(x))"""
    return np.sqrt(np.arctan(x))

# Altura del rectángulo para Acierto-y-Error (máximo de g en [0,1])
# arctan(x) es creciente y sqrt también => máximo en x=1
M = float(np.sqrt(np.arctan(1.0)))  # = sqrt(pi/4) = sqrt(pi)/2

# -------------------------
# Utilidades
# -------------------------
def resumen(vals):
    vals = np.asarray(vals, dtype=float)
    return float(np.mean(vals)), float(np.var(vals, ddof=1)), float(np.std(vals, ddof=1))

# -------------------------
# 1) Monte Carlo Crudo
# -------------------------
def mc_crudo(N, seed=123):
    rd.seed(seed)
    vals = []
    for _ in range(N):
        x = rd.random()          # U(0,1)
        vals.append(g(x))        # E[g(U)] = ∫ g
    media, var, std = resumen(vals)
    return media, var, std, vals

# -------------------------
# 2) Monte Carlo Antitético (sobre crudo)
# -------------------------
def mc_antitetico(N, seed=123):
    """
    Empareja U y 1-U, promedia f(U) y f(1-U) por par y luego promedia pares.
    N se fuerza a par para emparejar.
    """
    if N % 2: N += 1
    rd.seed(seed)
    m = N // 2
    vals = []
    for _ in range(m):
        u = rd.random()
        v = 1.0 - u
        vals.append(0.5*(g(u) + g(v)))
    media, var, std = resumen(vals)
    return media, var, std, vals

# -------------------------
# 3) Acierto-y-Error (Hit-or-Miss)
#     Integral = área bajo g = M * P(Y ≤ g(X)),
#     con X ~ U(0,1), Y ~ U(0,M)
# -------------------------
def mc_hit_or_miss(N, seed=123):
    rd.seed(seed)
    ests = []
    for _ in range(N):
        x = rd.random()
        y = rd.random() * M
        I = 1.0 if y <= g(x) else 0.0
        ests.append(M * I)  # cada indicador escala el área del rectángulo
    media, var, std = resumen(ests)
    return media, var, std, ests

# -------------------------
# 4) Acierto-y-Error Antitético
#     Empareja (X,Y) con (1-X, M-Y)
# -------------------------
def mc_hit_or_miss_antitetico(N, seed=123):
    if N % 2: N += 1
    rd.seed(seed)
    m = N // 2
    vals = []
    for _ in range(m):
        x = rd.random()
        y = rd.random() * M
        x_bar = 1.0 - x
        y_bar = M - y
        I1 = 1.0 if y     <= g(x)     else 0.0
        I2 = 1.0 if y_bar <= g(x_bar) else 0.0
        vals.append(M * 0.5*(I1 + I2))
    media, var, std = resumen(vals)
    return media, var, std, vals

# -------------------------
# 5) Ejecución y tiempos (ajusta N si quieres)
# -------------------------
N = 10_000

# MC Crudo
t0 = time.time()
res_c = mc_crudo(N)
t1 = time.time()

# MC Antitético (sobre crudo)
t2 = time.time()
res_a = mc_antitetico(N)
t3 = time.time()

# Hit-or-Miss (Acierto-y-Error)
t4 = time.time()
res_hm = mc_hit_or_miss(N)
t5 = time.time()

# Hit-or-Miss Antitético
t6 = time.time()
res_hm_a = mc_hit_or_miss_antitetico(N)
t7 = time.time()

# -------------------------
# 6) Resultados
# -------------------------
m_c, v_c, s_c, _ = res_c
m_a, v_a, s_a, _ = res_a
m_hm, v_hm, s_hm, _ = res_hm
m_hm_a, v_hm_a, s_hm_a, _ = res_hm_a

t_crudo         = t1 - t0
t_antitetico    = t3 - t2
t_hitmiss       = t5 - t4
t_hitmiss_anti  = t7 - t6

def eficiencia(t, var):  # time×variance (menor es mejor)
    return t * var

print("MC Crudo (estimador de la integral)")
print(f"Estimador = {m_c:.6f} | Var = {v_c:.6e} | Std = {s_c:.6e} | Tiempo = {t_crudo:.4f}s")

print("\nMC Antitético (sobre crudo)")
print(f"Estimador = {m_a:.6f} | Var = {v_a:.6e} | Std = {s_a:.6e} | Tiempo = {t_antitetico:.4f}s")
print(f"Reducción de varianza vs crudo = {100*(1 - v_a/v_c):.2f}%")
print(f"Eficiencia relativa (anti vs crudo) = {eficiencia(t_antitetico, v_a)/eficiencia(t_crudo, v_c):.4f} "
      f"(<1 ⇒ antitético más eficiente)")

print("\nAcierto-y-Error (Hit-or-Miss)")
print(f"Estimador = {m_hm:.6f} | Var = {v_hm:.6e} | Std = {s_hm:.6e} | Tiempo = {t_hitmiss:.4f}s")

print("\nAcierto-y-Error Antitético")
print(f"Estimador = {m_hm_a:.6f} | Var = {v_hm_a:.6e} | Std = {s_hm_a:.6e} | Tiempo = {t_hitmiss_anti:.4f}s")
print(f"% Reducción de varianza vs H-M = {100*(1 - v_hm_a/v_hm):.2f}%")
print(f"\nEficiencia relativa (H-M anti vs H-M) = {eficiencia(t_hitmiss_anti, v_hm_a)/eficiencia(t_hitmiss, v_hm):.4f} "
      f"(<1 ⇒ antitético más eficiente)")


MC Crudo (estimador de la integral)
Estimador = 0.630405 | Var = 4.269211e-02 | Std = 2.066207e-01 | Tiempo = 0.0162s

MC Antitético (sobre crudo)
Estimador = 0.629544 | Var = 2.755195e-03 | Std = 5.248995e-02 | Tiempo = 0.0164s
% Reducción de varianza vs crudo = 93.55%
Eficiencia relativa (anti vs crudo) = 0.0652 (<1 ⇒ antitético más eficiente)

Acierto-y-Error (Hit-or-Miss)
Estimador = 0.621688 | Var = 1.644771e-01 | Std = 4.055577e-01 | Tiempo = 0.0178s

Acierto-y-Error Antitético
Estimador = 0.628158 | Var = 4.776377e-02 | Std = 2.185492e-01 | Tiempo = 0.0167s
% Reducción de varianza vs H-M = 70.96%

Eficiencia relativa (H-M anti vs H-M) = 0.2715 (<1 ⇒ antitético más eficiente)
